# Phase 1: Exploratory Data Analysis
## Section 1: Dataset Overview & Data Quality Assessment

**Project:** Tamil Nadu Next-Day Peak Electricity Demand Forecasting  
**Repository Branch:** `track-a-modelling`  
**Target Variable:** `tn_peak_demand_mw`  
**Production Dataset:** `data/master.csv`  

### Objective of this Section
Perform an initial, strictly non-mutating inspection of `data/master.csv` to verify:
1. Dimensionality, temporal boundaries, and schema integrity.
2. Data quality, including missingness, duplicate timestamps, and chronological continuity.
3. Baseline descriptive statistical properties of the primary target variable (`tn_peak_demand_mw`).

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

# Display formatting options
pd.set_option("display.max_columns", 60)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

# Resolve dataset path flexibly whether executed from repository root or notebooks/ directory
DATA_PATH = Path("../data/master.csv") if Path("../data/master.csv").exists() else Path("data/master.csv")
print(f"Loading dataset from: {DATA_PATH.resolve()}")

### 1. Dataset Ingestion & Shape
Load `data/master.csv` without modifying it and parse the `date` column.

In [ ]:
df = pd.read_csv(DATA_PATH, parse_dates=["date"])

n_rows, n_cols = df.shape
print("=" * 60)
print(f"Dataset Shape: {n_rows:,} rows, {n_cols} columns")
print("=" * 60)
df.head(5)

### 2. Temporal Range & Continuity
Evaluate the temporal coverage, chronological ordering, duplicate date occurrences, and calendar continuity.

In [ ]:
min_date = df["date"].min()
max_date = df["date"].max()
total_calendar_days = (max_date - min_date).days + 1
observed_days = df["date"].nunique()
missing_days_count = total_calendar_days - observed_days

print("=" * 60)
print("TEMPORAL RANGE & CONTINUITY")
print("=" * 60)
print(f"Earliest Observation Date : {min_date.strftime('%Y-%m-%d')} ({min_date.strftime('%A')})")
print(f"Latest Observation Date   : {max_date.strftime('%Y-%m-%d')} ({max_date.strftime('%A')})")
print(f"Total Calendar Span (Days): {total_calendar_days:,}")
print(f"Observed Records (Days)   : {observed_days:,}")
print(f"Missing Calendar Days     : {missing_days_count:,} ({missing_days_count / total_calendar_days * 100:.2f}%)")
print("=" * 60)

# Check duplicate dates
duplicate_dates = df["date"].duplicated().sum()
print(f"Duplicate Date Entries    : {duplicate_dates}")
assert duplicate_dates == 0, "Data quality violation: duplicate dates found!"

# Identify distribution of missing calendar dates
full_span = pd.date_range(min_date, max_date, freq="D")
missing_dates = full_span.difference(df["date"])
if len(missing_dates) > 0:
    missing_series = pd.Series(missing_dates)
    print(f"\nMissing calendar dates by year:")
    print(missing_series.dt.year.value_counts().sort_index().to_string())

### 3. Schema & Column Data Types
Examine column data types, non-null counts, null percentages, and sample values across the 54 columns.

In [ ]:
schema_summary = pd.DataFrame({
    "Column": df.columns,
    "Dtype": df.dtypes.astype(str).values,
    "Non-Null Count": df.notna().sum().values,
    "Null Count": df.isna().sum().values,
    "Null %": (df.isna().mean() * 100).round(2).values,
    "Sample Value": [df[col].dropna().iloc[0] if df[col].notna().any() else np.nan for col in df.columns]
})

schema_summary

### 4. Data Quality: Missing Value Assessment
Isolate all columns with missing values and evaluate whether nulls reflect expected lag boundaries or anomalous missingness.

In [ ]:
missing_df = schema_summary[schema_summary["Null Count"] > 0].copy()
missing_df = missing_df.sort_values(by="Null Count", ascending=False).reset_index(drop=True)

print(f"Total columns with missing values: {len(missing_df)} out of {n_cols}")
print("-" * 60)
missing_df[["Column", "Dtype", "Null Count", "Null %"]]

### 5. Primary Target Variable (`tn_peak_demand_mw`) Descriptive Statistics
Compute central tendency, spread, quantiles, and distribution shape metrics for Tamil Nadu peak demand.

In [ ]:
target_col = "tn_peak_demand_mw"
s = df[target_col]

stats_dict = {
    "Metric": [
        "Observation Count", "Mean (MW)", "Std Dev (MW)", "Min (MW)",
        "1st Percentile (MW)", "5th Percentile (MW)", "25th Percentile / Q1 (MW)",
        "Median / Q2 (MW)", "75th Percentile / Q3 (MW)",
        "95th Percentile (MW)", "99th Percentile (MW)", "Max (MW)",
        "Interquartile Range (IQR, MW)", "Skewness", "Kurtosis"
    ],
    "Value": [
        f"{s.count():,}",
        f"{s.mean():,.2f}",
        f"{s.std():,.2f}",
        f"{s.min():,.2f}",
        f"{s.quantile(0.01):,.2f}",
        f"{s.quantile(0.05):,.2f}",
        f"{s.quantile(0.25):,.2f}",
        f"{s.median():,.2f}",
        f"{s.quantile(0.75):,.2f}",
        f"{s.quantile(0.95):,.2f}",
        f"{s.quantile(0.99):,.2f}",
        f"{s.max():,.2f}",
        f"{(s.quantile(0.75) - s.quantile(0.25)):,.2f}",
        f"{s.skew():.3f}",
        f"{s.kurtosis():.3f}"
    ]
}

target_stats_df = pd.DataFrame(stats_dict)
target_stats_df

### 6. Section 1 Summary & Data Quality Verdict

#### Key Dataset Facts:
1. **Dimensionality:** 4,897 daily observations and 54 columns.
2. **Temporal Coverage:** 2013-01-02 to 2026-11-09 (span of 5,060 calendar days).
3. **Unique Dates:** Zero duplicate date entries.
4. **Completeness of Core Target:** Primary target `tn_peak_demand_mw` is 100% complete with 0 missing values.
5. **Data Quality Profile:**
   - 163 calendar dates are missing across the entire 14-year span (predominantly in 2013–2014 historical records).
   - Missing values in columns are exclusively confined to `holiday_name` (structural sparsity on non-holidays) and lag/rolling feature warm-up windows.
6. **Target Distribution Profile:**
   - Minimum peak demand: 7,359.00 MW
   - Median peak demand: 14,066.00 MW
   - Mean peak demand: 14,332.55 MW
   - Maximum peak demand: 21,724.00 MW
   - IQR: 3,237.00 MW (Q1: 12,561.00 MW, Q3: 15,798.00 MW)

---
# Section 2 — Target Time-Series and Long-Term Trend

This section analyzes `tn_peak_demand_mw` across the entire historical record (2013 to 2026) to:
1. Track annual mean and maximum peak demand growth.
2. Compute year-over-year (YoY) percentage changes.
3. Examine monthly demand averages across years.
4. Separate long-term secular growth from seasonal cycles.

In [ ]:
# Extract temporal dimensions
df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month

# Annual aggregation
annual_stats = df.groupby('year')['tn_peak_demand_mw'].agg([
    ('record_count', 'count'),
    ('mean_mw', 'mean'),
    ('max_mw', 'max'),
    ('min_mw', 'min'),
    ('std_mw', 'std')
]).reset_index()

# Compute Year-over-Year percentage change in mean and max demand
annual_stats['yoy_mean_growth_pct'] = annual_stats['mean_mw'].pct_change() * 100
annual_stats['yoy_max_growth_pct'] = annual_stats['max_mw'].pct_change() * 100

print("=" * 75)
print("ANNUAL PEAK DEMAND SUMMARY & YEAR-OVER-YEAR GROWTH")
print("=" * 75)
display_cols = ['year', 'record_count', 'mean_mw', 'max_mw', 'min_mw', 'yoy_mean_growth_pct', 'yoy_max_growth_pct']
display_df = annual_stats[display_cols].copy()
display_df.columns = ['Year', 'Days', 'Mean (MW)', 'Max (MW)', 'Min (MW)', 'YoY Mean (%)', 'YoY Max (%)']
display_df

In [ ]:
# Monthly profile across the entire dataset
monthly_summary = df.groupby('month')['tn_peak_demand_mw'].agg([
    ('record_count', 'count'),
    ('mean_mw', 'mean'),
    ('std_mw', 'std'),
    ('min_mw', 'min'),
    ('max_mw', 'max')
]).reset_index()

month_names = {1: 'January', 2: 'February', 3: 'March', 4: 'April', 5: 'May', 6: 'June',
               7: 'July', 8: 'August', 9: 'September', 10: 'October', 11: 'November', 12: 'December'}
monthly_summary['month_name'] = monthly_summary['month'].map(month_names)

print("=" * 75)
print("MONTHLY AVERAGE PEAK ELECTRICITY DEMAND (2013-2026)")
print("=" * 75)
monthly_disp = monthly_summary[['month', 'month_name', 'record_count', 'mean_mw', 'std_mw', 'min_mw', 'max_mw']].copy()
monthly_disp.columns = ['Month', 'Name', 'Days', 'Mean (MW)', 'Std Dev (MW)', 'Min (MW)', 'Max (MW)']
monthly_disp

In [ ]:
# Visual representation of Long-Term Trend and Annual Profile
try:
    import matplotlib.pyplot as plt
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10))
    
    # Full Time Series
    ax1.plot(df['date'], df['tn_peak_demand_mw'], color='#1f77b4', lw=0.8, alpha=0.85, label='Daily Peak Demand (MW)')
    ax1.set_title('Tamil Nadu Daily Peak Electricity Demand (2013 - 2026)', fontsize=14, fontweight='bold')
    ax1.set_ylabel('Peak Demand (MW)', fontsize=12)
    ax1.grid(True, linestyle='--', alpha=0.5)
    ax1.legend(loc='upper left')
    
    # Annual Mean and Max Bars
    years = annual_stats['year']
    w = 0.35
    ax2.bar(years - w/2, annual_stats['mean_mw'], width=w, label='Annual Mean (MW)', color='#2ca02c', alpha=0.85)
    ax2.bar(years + w/2, annual_stats['max_mw'], width=w, label='Annual Maximum (MW)', color='#d62728', alpha=0.85)
    ax2.set_title('Annual Mean vs. Maximum Peak Demand (2013 - 2026)', fontsize=14, fontweight='bold')
    ax2.set_xlabel('Year', fontsize=12)
    ax2.set_ylabel('Demand (MW)', fontsize=12)
    ax2.set_xticks(years)
    ax2.set_xticklabels(years, rotation=45)
    ax2.grid(True, linestyle='--', alpha=0.5)
    ax2.legend(loc='upper left')
    
    plt.tight_layout()
    plt.show()
except ImportError:
    print("Note: matplotlib not found; visual summary rendered in tabular format above.")
    print("Long-Term Growth Summary:")
    print(f"  2013 Mean Demand: {annual_stats.loc[annual_stats['year']==2013, 'mean_mw'].values[0]:,.1f} MW")
    print(f"  2026 Mean Demand: {annual_stats.loc[annual_stats['year']==2026, 'mean_mw'].values[0]:,.1f} MW")
    overall_growth = ((annual_stats.loc[annual_stats['year']==2026, 'mean_mw'].values[0] / annual_stats.loc[annual_stats['year']==2013, 'mean_mw'].values[0]) - 1) * 100
    print(f"  Total Mean Demand Growth: +{overall_growth:.2f}%")

### Section 2 Observations: Long-Term Trend vs. Seasonal Variation

1. **Secular Demand Growth:**
   - Tamil Nadu's annual mean peak demand expanded from **10,911.39 MW in 2013** to **18,986.64 MW in 2026** (a net increase of **+74.01%**, or ~620 MW per year on average).
   - The annual maximum peak demand rose from **12,218.00 MW in 2013** to an all-time record of **21,724.00 MW in July 2026** (+77.80%).
   - Growth was interrupted only in **2020 (-7.35% YoY mean decline)** due to widespread COVID-19 commercial and industrial lockdowns, followed by a swift rebound in 2021 (+7.49%) and sustained acceleration through 2023–2026.

2. **Distinct Seasonal Variations:**
   - Superimposed upon this multi-year secular trend is a recurring annual seasonal wave.
   - **Summer Highs (March–May):** March and April exhibit the highest monthly averages (15,319 MW and 15,137 MW) driven by daytime and evening cooling demand.
   - **Mid-Year Secondary Surge (June–September):** July reaches 14,742 MW average with the historical single-day record peak of 21,724 MW.
   - **Monsoon Lows (October–December):** Demand drops to its yearly nadir in November (13,012 MW) and December (13,177 MW), aligning with cooler temperatures and the Northeast monsoon.

---
# Section 3 — Calendar and Seasonal Effects

This section quantifies calendar-driven shifts in Tamil Nadu peak demand:
1. Day of Week profile (Monday through Sunday).
2. Weekend vs. Weekday demand contrast.
3. Public Holiday vs. Non-Holiday impact.
4. Pongal festival window effect (Jan 13–17).

In [ ]:
# 1. Day of Week Profile
dow_names = {0: 'Monday', 1: 'Tuesday', 2: 'Wednesday', 3: 'Thursday', 4: 'Friday', 5: 'Saturday', 6: 'Sunday'}
dow_summary = df.groupby('dow')['tn_peak_demand_mw'].agg([
    ('count', 'count'),
    ('mean_mw', 'mean'),
    ('std_mw', 'std'),
    ('min_mw', 'min'),
    ('max_mw', 'max')
]).reset_index()
dow_summary['day_name'] = dow_summary['dow'].map(dow_names)

# Difference relative to midweek baseline (Wednesday)
wed_mean = dow_summary.loc[dow_summary['dow'] == 2, 'mean_mw'].values[0]
dow_summary['diff_vs_wed_mw'] = dow_summary['mean_mw'] - wed_mean
dow_summary['pct_diff_vs_wed'] = (dow_summary['diff_vs_wed_mw'] / wed_mean) * 100

print("=" * 75)
print("DAY OF WEEK PEAK DEMAND PROFILE")
print("=" * 75)
dow_disp = dow_summary[['dow', 'day_name', 'count', 'mean_mw', 'std_mw', 'diff_vs_wed_mw', 'pct_diff_vs_wed']].copy()
dow_disp.columns = ['DOW', 'Day Name', 'Days', 'Mean (MW)', 'Std Dev (MW)', 'Diff vs Wed (MW)', 'Diff vs Wed (%)']
dow_disp

In [ ]:
# 2. Weekend vs Weekday, Holiday vs Non-Holiday, and Pongal Window Profiles

# Weekend
wknd = df.groupby('is_weekend')['tn_peak_demand_mw'].agg(['count', 'mean', 'std', 'min', 'max']).reset_index()
wknd['segment'] = wknd['is_weekend'].map({0: 'Weekday (Mon-Fri)', 1: 'Weekend (Sat-Sun)'})

# Holiday
holi = df.groupby('is_holiday')['tn_peak_demand_mw'].agg(['count', 'mean', 'std', 'min', 'max']).reset_index()
holi['segment'] = holi['is_holiday'].map({0: 'Non-Holiday', 1: 'Gazetted Holiday'})

# Pongal Window
pong = df.groupby('pongal_window')['tn_peak_demand_mw'].agg(['count', 'mean', 'std', 'min', 'max']).reset_index()
pong['segment'] = pong['pongal_window'].map({0: 'Non-Pongal Days', 1: 'Pongal Window (Jan 13-17)'})

calendar_comparisons = pd.concat([
    wknd[['segment', 'count', 'mean', 'std', 'min', 'max']],
    holi[['segment', 'count', 'mean', 'std', 'min', 'max']],
    pong[['segment', 'count', 'mean', 'std', 'min', 'max']]
], ignore_index=True)

calendar_comparisons.columns = ['Calendar Segment', 'Days Count', 'Mean (MW)', 'Std Dev (MW)', 'Min (MW)', 'Max (MW)']
print("=" * 75)
print("CALENDAR SEGMENT COMPARISONS")
print("=" * 75)
calendar_comparisons

In [ ]:
# Visualizing Calendar Effects
try:
    import matplotlib.pyplot as plt
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # DOW Bar Chart
    colors_dow = ['#1f77b4']*5 + ['#ff7f0e', '#d62728']
    ax1.bar(dow_summary['day_name'], dow_summary['mean_mw'], color=colors_dow, alpha=0.85)
    ax1.set_title('Average Peak Demand by Day of Week', fontsize=12, fontweight='bold')
    ax1.set_ylabel('Mean Peak Demand (MW)')
    ax1.set_ylim(12000, 15500)
    ax1.grid(True, linestyle='--', alpha=0.5)
    
    # Calendar Comparison Bar Chart
    labels = ['Weekday', 'Weekend', 'Non-Holiday', 'Holiday', 'Regular', 'Pongal']
    means = [
        wknd.loc[wknd['is_weekend']==0, 'mean'].values[0],
        wknd.loc[wknd['is_weekend']==1, 'mean'].values[0],
        holi.loc[holi['is_holiday']==0, 'mean'].values[0],
        holi.loc[holi['is_holiday']==1, 'mean'].values[0],
        pong.loc[pong['pongal_window']==0, 'mean'].values[0],
        pong.loc[pong['pongal_window']==1, 'mean'].values[0]
    ]
    ax2.bar(labels, means, color=['#2ca02c', '#d62728', '#2ca02c', '#d62728', '#2ca02c', '#9467bd'], alpha=0.85)
    ax2.set_title('Calendar Segment Impact Comparison', fontsize=12, fontweight='bold')
    ax2.set_ylabel('Mean Peak Demand (MW)')
    ax2.set_ylim(11000, 15500)
    ax2.grid(True, linestyle='--', alpha=0.5)
    
    plt.tight_layout()
    plt.show()
except ImportError:
    print("Matplotlib visual plot skipped (not installed). Tabular metrics rendered above.")

### Section 3 Observations: Observed Calendar Patterns

1. **Day of Week Uniformity vs. Sunday Drop:**
   - Tuesday through Friday form a tightly clustered plateau between **14,595 MW and 14,625 MW** (variance < 30 MW across these 4 working days).
   - Monday demand is marginally lower (**14,418 MW**, -1.24% vs. Wednesday) as industrial operations ramp up.
   - Saturday demand remains surprisingly resilient (**14,403 MW**, -1.34% vs. Wednesday), reflecting partial-day industrial shifts and active retail.
   - **Sunday drops sharply to 13,074 MW** (a severe reduction of **-1,525 MW / -10.45%** vs. Wednesday), representing the single largest recurring weekly demand suppression.

2. **Weekend vs. Weekday Aggregate:**
   - Weekday average: **14,572.38 MW** (3,493 days).
   - Weekend average: **13,735.88 MW** (1,404 days) — an aggregate drop of **-836.50 MW (-5.74%)**.

3. **Gazetted Holiday Suppression:**
   - Non-holiday average: **14,385.27 MW** (4,580 days).
   - Holiday average: **13,570.95 MW** (317 days) — an average drop of **-814.31 MW (-5.66%)**.

4. **Pongal Festival Window (Jan 13–17):**
   - Across all 65 observed Pongal festival days, peak demand averaged **12,315.91 MW**, compared to **14,359.68 MW** on non-Pongal days.
   - This constitutes an acute reduction of **-2,043.77 MW (-14.23%)**, making the Pongal festival window the most impactful multi-day demand suppression event on the Tamil Nadu grid.

---
# Section 4 — Weather Relationship Analysis

> **CRITICAL MODELLING NOTICE:**
> The weather variables in `data/weather_daily.csv` (and joined into `data/master.csv`) are **retrospective ERA5 reanalysis and historical actual observations**.
> In an operational next-day forecasting pipeline (issued at 20:00 IST on Day $D$ for Day $D+1$),
> **actual target-day weather is NOT available**; only a numerical weather prediction (NWP) forecast is available.
> The relationships analyzed below represent historical retrospective correlations.

Features analyzed:
`t_mean`, `t_max`, `t_min`, `rh_mean`, `app_t_mean`, `app_t_max`, `hours_above_32`, `cdd`, `cdd_max`, `cdd_apparent`, `thi`.

In [ ]:
weather_cols = [
    't_mean', 't_max', 't_min', 'rh_mean',
    'app_t_mean', 'app_t_max', 'hours_above_32',
    'cdd', 'cdd_max', 'cdd_apparent', 'thi'
]

# 1. Correlation table with tn_peak_demand_mw
weather_corr = df[['tn_peak_demand_mw'] + weather_cols].corr()['tn_peak_demand_mw'].drop('tn_peak_demand_mw')
weather_corr_table = pd.DataFrame({
    'Weather Feature': weather_corr.index,
    'Correlation with Target': weather_corr.values,
    'Abs Correlation': weather_corr.abs().values
}).sort_values(by='Abs Correlation', ascending=False).reset_index(drop=True)

print("=" * 75)
print("CORRELATION BETWEEN TN PEAK DEMAND AND WEATHER FEATURES (RETROSPECTIVE)")
print("=" * 75)
weather_corr_table[['Weather Feature', 'Correlation with Target']]

In [ ]:
# 2. Weather Correlation Matrix
weather_corr_matrix = df[weather_cols].corr()

print("=" * 75)
print("INTER-FEATURE WEATHER CORRELATION MATRIX")
print("=" * 75)
weather_corr_matrix.round(3)

In [ ]:
# Visualizations: Weather Heatmap & Scatterplots for the 4 Strongest Relationships
top_4_weather = weather_corr_table['Weather Feature'].head(4).tolist()
print(f"Top 4 Strongest Weather Features: {top_4_weather}")

try:
    import matplotlib.pyplot as plt
    import seaborn as sns
    
    # Heatmap
    plt.figure(figsize=(10, 8))
    sns.heatmap(weather_corr_matrix, annot=True, fmt=".2f", cmap='coolwarm', cbar=True, square=True)
    plt.title('Inter-Weather Correlation Heatmap', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    # Scatter plots for Top 4 Features
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    for ax, feat in zip(axes.flatten(), top_4_weather):
        ax.scatter(df[feat], df['tn_peak_demand_mw'], alpha=0.2, color='#1f77b4', s=10)
        # Add trend line
        m, b = np.polyfit(df[feat].dropna(), df.loc[df[feat].notna(), 'tn_peak_demand_mw'], 1)
        ax.plot(df[feat], m*df[feat] + b, color='red', lw=1.5, label=f'r = {df[feat].corr(df["tn_peak_demand_mw"]):.2f}')
        ax.set_title(f'Peak Demand vs. {feat}', fontsize=12, fontweight='bold')
        ax.set_xlabel(feat, fontsize=10)
        ax.set_ylabel('Peak Demand (MW)', fontsize=10)
        ax.grid(True, linestyle='--', alpha=0.5)
        ax.legend(loc='upper left')
    plt.tight_layout()
    plt.show()
except ImportError:
    print("Matplotlib/Seaborn not installed. Showing top 4 weather feature distributions:")
    for feat in top_4_weather:
        r = df[feat].corr(df['tn_peak_demand_mw'])
        print(f"  {feat:<16}: r = {r:+.3f} | Mean = {df[feat].mean():.2f} | Min = {df[feat].min():.2f} | Max = {df[feat].max():.2f}")

### Section 4 Observations: Weather Interactions

1. **Dominance of Maximum Heat Metrics:**
   - `t_max` (**r = +0.397**) and `cdd_max` (**r = +0.397**) exhibit the strongest positive linear correlations with daily peak demand.
   - `t_mean` (**r = +0.379**) and `app_t_max` (**r = +0.374**) follow closely.
   - Daytime extreme temperature triggers immediate commercial, agricultural, and residential air conditioning / cooling loads that coincide with or drive the daily grid peak.

2. **Negative Correlation with Relative Humidity:**
   - `rh_mean` exhibits a negative correlation (**r = -0.268**) with peak demand.
   - High relative humidity in Tamil Nadu primarily coincides with active rainy/monsoon periods (Northeast monsoon in Oct–Dec) which suppress temperatures and reduce overall agricultural pumping and cooling demand.

3. **Extreme Weather Collinearity:**
   - `t_max` and `cdd_max` are virtually identical (**r = 0.99997**), because in Tamil Nadu daily maximum temperature almost always exceeds the 25°C base.
   - `t_mean` and `cdd` are near-identical (**r = 0.992**).
   - `app_t_mean` and `cdd_apparent` have **r = 0.999**.
   - Including all these variants simultaneously in linear models will introduce severe multicollinearity without providing unique signal.

---
# Section 5 — Feature Correlation and Statistical Profiling

This section conducts concise profiling across all 54 features:
1. Top positive and negative correlations with `tn_peak_demand_mw`.
2. Identification of highly collinear feature pairs ($|r| \ge 0.85$).
3. Structural vs. redundant feature differentiation.
4. Statistical evaluation of long-term distribution shift (Early 2013–2015 vs. Recent 2024–2026).

In [ ]:
num_df = df.select_dtypes(include=[np.number])
target_corr = num_df.corr()['tn_peak_demand_mw'].drop('tn_peak_demand_mw').sort_values(ascending=False)

print("=" * 75)
print("TOP 10 POSITIVELY CORRELATED NUMERICAL FEATURES")
print("=" * 75)
top_pos = pd.DataFrame({
    'Feature': target_corr.head(10).index,
    'Pearson Correlation': target_corr.head(10).values
})
print(top_pos.to_string(index=False))

print("\n" + "=" * 75)
print("TOP 5 NEGATIVELY CORRELATED NUMERICAL FEATURES")
print("=" * 75)
top_neg = pd.DataFrame({
    'Feature': target_corr.tail(5).index,
    'Pearson Correlation': target_corr.tail(5).values
})
print(top_neg.to_string(index=False))

In [ ]:
# Collinear Pairs with |r| >= 0.85
full_corr = num_df.corr()
cols = full_corr.columns
collinear_pairs = []

for i in range(len(cols)):
    for j in range(i + 1, len(cols)):
        c_val = full_corr.iloc[i, j]
        if abs(c_val) >= 0.85:
            collinear_pairs.append({
                'Feature_1': cols[i],
                'Feature_2': cols[j],
                'Correlation': round(c_val, 4),
                'Abs_Corr': round(abs(c_val), 4)
            })

collinear_df = pd.DataFrame(collinear_pairs).sort_values(by='Abs_Corr', ascending=False).reset_index(drop=True)

print("=" * 75)
print(f"HIGHLY CORRELATED FEATURE PAIRS (|r| >= 0.85): {len(collinear_df)} PAIRS FOUND")
print("=" * 75)
print("Sample of Top 20 Highly Collinear Pairs:")
collinear_df.head(20)

In [ ]:
# Evaluating Long-Term Distribution Shift: Early (2013-2015) vs Recent (2024-2026)
early_mask = df['year'].between(2013, 2015)
recent_mask = df['year'].between(2024, 2026)

early_y = df.loc[early_mask, 'tn_peak_demand_mw']
recent_y = df.loc[recent_mask, 'tn_peak_demand_mw']

dist_shift = pd.DataFrame({
    'Statistic': ['Obs Count', 'Mean (MW)', 'Std Dev (MW)', 'Min (MW)', 'Q1 / 25% (MW)', 'Median (MW)', 'Q3 / 75% (MW)', 'Max (MW)', 'IQR (MW)'],
    'Early (2013-2015)': [
        early_y.count(), early_y.mean(), early_y.std(), early_y.min(),
        early_y.quantile(0.25), early_y.median(), early_y.quantile(0.75),
        early_y.max(), early_y.quantile(0.75) - early_y.quantile(0.25)
    ],
    'Recent (2024-2026)': [
        recent_y.count(), recent_y.mean(), recent_y.std(), recent_y.min(),
        recent_y.quantile(0.25), recent_y.median(), recent_y.quantile(0.75),
        recent_y.max(), recent_y.quantile(0.75) - recent_y.quantile(0.25)
    ]
})

dist_shift['Absolute Shift'] = dist_shift['Recent (2024-2026)'] - dist_shift['Early (2013-2015)']
dist_shift['Percentage Shift (%)'] = (dist_shift['Absolute Shift'] / dist_shift['Early (2013-2015)']) * 100

print("=" * 75)
print("TARGET DISTRIBUTION SHIFT: EARLY (2013-2015) VS RECENT (2024-2026)")
print("=" * 75)
dist_shift

### Section 5 Observations: Statistical Profiling & Distribution Shift

1. **Top Predictive Correlations:**
   - `tn_energy_gwh` shows a massive **r = +0.971** with peak demand (reflecting overall daily load volume), but same-day energy cannot be used as an input feature for next-day forecasting because it is unobserved.
   - Autoregressive lags (`peak_roll_mean_7` r = +0.929, `peak_lag_1` r = +0.927, `peak_lag_7` r = +0.907) dominate all other predictors.
   - Notice that under the verified $H=2$ operational constraint, `peak_lag_1` and `energy_lag_1` are unavailable; the model will rely on `peak_lag_2` (**r = +0.894**) and `peak_roll_mean_7` lagged to $T-2$.

2. **Negative Correlations:**
   - `sr_peak_shortage_mw_lag1` (**r = -0.475**): Historically, higher Southern Region shortages occurred during 2013–2015 when grid infrastructure was constrained and demand was structurally lower.
   - `rh_mean` (**r = -0.268**) and calendar indicators (`is_weekend` r = -0.153, `dow` r = -0.128).

3. **Collinearity Breakdown (156 pairs with $|r| \ge 0.85$):**
   - **Redundant Weather Transformations:** `t_max` vs. `cdd_max` (r = 1.000), `app_t_mean` vs. `cdd_apparent` (r = 0.999), `t_mean` vs. `cdd` (r = 0.992). Feature selection will be required to prune duplicates.
   - **Autoregressive Co-Movement:** Rolling averages and lags (`peak_roll_mean_30` vs. `energy_roll_mean_30` r = 0.980; `peak_lag_14` vs. `energy_lag_14` r = 0.972). This is expected physical co-movement, not synthetic leakage.

4. **Severe Long-Term Distribution Shift:**
   - Between 2013–2015 and 2024–2026, the target distribution underwent substantial non-stationary expansion:
     - **Mean load shifted upward by +51.66% (+5,982.68 MW)**.
     - **Median load shifted upward by +51.32% (+5,966.00 MW)**.
     - **Standard deviation nearly doubled (+91.12%, from 1,008 MW to 1,927 MW)**.
     - **Interquartile range (IQR) more than doubled (+104.16%, from 1,351 MW to 2,758 MW)**.
   - *Critical Modelling Takeaway:* Standard random k-fold cross-validation is completely invalid. Time-series temporal splits (walk-forward or expanding window cross-validation) must be used.

---
# Section 6 — High-Demand and Extreme-Day Analysis

Operationally, forecasting errors during extreme demand peaks carry the highest economic and grid reliability penalties.
This section analyzes:
1. The 95th percentile peak demand threshold across the full dataset.
2. Temporal distribution of high-demand days by year and by month.
3. The Top 10 highest-demand days in Tamil Nadu history.

In [ ]:
# 1. Calculate 95th percentile threshold
threshold_95 = df['tn_peak_demand_mw'].quantile(0.95)
high_demand_df = df[df['tn_peak_demand_mw'] >= threshold_95].copy()
n_high = len(high_demand_df)
pct_high = (n_high / len(df)) * 100

print("=" * 75)
print(f"TOP 5% HIGH-DEMAND THRESHOLD (95th PERCENTILE): {threshold_95:,.2f} MW")
print(f"Total High-Demand Days: {n_high} days ({pct_high:.2f}% of full dataset)")
print("=" * 75)

In [ ]:
# 2. High-Demand Days Distribution by Year and Month
hd_by_year = high_demand_df['year'].value_counts().sort_index().reset_index()
hd_by_year.columns = ['Year', 'High-Demand Days Count']
hd_by_year['Pct of High-Demand Days (%)'] = (hd_by_year['High-Demand Days Count'] / n_high) * 100

hd_by_month = high_demand_df['month'].value_counts().sort_index().reset_index()
hd_by_month.columns = ['Month', 'High-Demand Days Count']
hd_by_month['Month Name'] = hd_by_month['Month'].map(month_names)
hd_by_month['Pct of High-Demand Days (%)'] = (hd_by_month['High-Demand Days Count'] / n_high) * 100

print("HIGH-DEMAND DAYS BY YEAR:")
print(hd_by_year.to_string(index=False))

print("\nHIGH-DEMAND DAYS BY MONTH:")
print(hd_by_month[['Month', 'Month Name', 'High-Demand Days Count', 'Pct of High-Demand Days (%)']].to_string(index=False))

In [ ]:
# 3. Top 10 Highest-Demand Days in History
top10 = df.sort_values(by='tn_peak_demand_mw', ascending=False).head(10).copy()
top10['day_name'] = top10['dow'].map(dow_names)

top10_table = pd.DataFrame({
    'Rank': range(1, 11),
    'Date': top10['date'].dt.strftime('%Y-%m-%d'),
    'Day': top10['day_name'],
    'Peak Demand (MW)': top10['tn_peak_demand_mw'].apply(lambda x: f"{x:,.1f}"),
    'Energy Met (GWh)': top10['tn_energy_gwh'].apply(lambda x: f"{x:,.2f}"),
    'Max Temp (°C)': top10['t_max'].apply(lambda x: f"{x:.2f}"),
    'CDD Max': top10['cdd_max'].apply(lambda x: f"{x:.2f}"),
    'Is Holiday': top10['is_holiday'].astype(int)
})

print("=" * 75)
print("TOP 10 HIGHEST PEAK DEMAND DAYS IN TAMIL NADU (2013-2026)")
print("=" * 75)
top10_table

### Section 6 Observations: High-Demand Clustering

1. **Heavy Temporal Concentration in Recent Years:**
   - Out of the 245 high-demand days ($\ge$ 19,126.60 MW):
     - **2013–2022 (10 years):** 0 days.
     - **2023:** 1 day (0.4%).
     - **2024:** 42 days (17.1%).
     - **2025:** 31 days (12.7%).
     - **2026:** **171 days (69.8%)**.
   - Nearly 70% of all historical extreme peak days occurred in the first 10 months of 2026, highlighting rapid load electrification and summer temperature anomalies.

2. **Seasonal Distribution of Peaks:**
   - **April dominates:** 61 high-demand days (24.9%), representing peak heat before the Southwest monsoon.
   - **July is secondary peak:** 36 high-demand days (14.7%), where high humidity and agricultural pumping demand compound.
   - March (28 days), May (27 days), June (27 days), August (26 days), and September (25 days) form the broad extreme-demand window.

3. **Top 10 Day Characteristics:**
   - **100% of the top 10 days occurred in 2026** (specifically between July 13 and September 10, 2026).
   - Peak load reached **21,724 MW on Tuesday, 2026-07-14**.
   - All top 10 days occurred on regular working weekdays (Monday, Tuesday, Wednesday, Thursday) with zero holidays.
   - All coincided with statewide maximum temperatures between **35.0°C and 37.5°C**.

---
# Key Findings for Modelling

Synthesizing the empirical observations from Sections 1 through 6, the following eight core findings establish the structural requirements for Phase 2 modelling:

### 1. Long-Term Demand Growth
- Mean peak demand grew by **+74.01% (+8,075 MW)** from 2013 (10,911 MW) to 2026 (18,987 MW), averaging ~620 MW of secular structural growth annually.
- Models must incorporate explicit structural growth features (e.g. `trend`, rolling annual averages, or differencing) to avoid under-predicting future horizons.

### 2. Major Seasonal Patterns
- A dual-peak seasonal cycle is evident:
  1. Primary summer peak in **March–April** (monthly averages exceeding 15,300 MW).
  2. Secondary monsoon break / agricultural peak in **July–September** (reaching all-time records up to 21,724 MW).
- A distinct seasonal nadir occurs during the Northeast monsoon in **November** (13,012 MW) and **December** (13,177 MW).
- Annual Fourier seasonality harmonics (`sin1`, `cos1`, `sin2`, `cos2`) capture this smooth seasonal oscillation.

### 3. Calendar Effects
- **Sunday Suppression:** Sunday demand drops by **-1,525 MW (-10.45%)** relative to midweek working days, representing a massive and highly reliable weekly drop.
- **Festival Drops:** The multi-day **Pongal festival window** (Jan 13–17) induces an acute average reduction of **-2,044 MW (-14.23%)**.
- **Gazetted Holidays:** Induce an average reduction of **-814 MW (-5.66%)**.
- Calendar categorical indicators (`dow`, `is_weekend`, `is_holiday`, `pongal_window`) are essential linear/tree features.

### 4. Strongest Weather Relationships
- Maximum daily temperature (`t_max`, r = +0.397) and cooling degree days (`cdd_max`, r = +0.397) have the strongest positive correlation with grid peak.
- Mean temperature (`t_mean`, r = +0.379) and apparent maximum temperature (`app_t_max`, r = +0.374) are strong secondary drivers.
- Relative humidity is inversely correlated (`rh_mean`, r = -0.268) due to monsoon rainfall cooling.

### 5. Evidence of Distribution Shift & Non-Stationarity
- A comparison between early (2013–2015) and recent (2024–2026) periods confirms severe non-stationarity:
  - Mean load increased by **+51.7%**.
  - Load volatility (standard deviation) increased by **+91.1%**.
  - Interquartile range (IQR) more than doubled (**+104.2%**).
- **Implication:** Random k-fold cross-validation is invalid. Temporal walk-forward validation (e.g. expanding window or rolling origin) is strictly required.

### 6. Highly Correlated & Redundant Features
- **156 feature pairs** exhibit $|r| \ge 0.85$.
- Key redundancies:
  - `t_max` vs. `cdd_max` (r = 1.000) and `t_mean` vs. `cdd` (r = 0.992) — redundant weather transformations.
  - `app_t_mean` vs. `cdd_apparent` (r = 0.999) and `thi` (r = 0.988).
  - Same-day energy met (`tn_energy_gwh`, r = +0.971) cannot be used as an input feature for next-day forecasting because it is unobserved at forecast time.
- Regularized models (Lasso / Ridge / ElasticNet) or explicit feature selection must be employed.

### 7. Missing Calendar Dates and Structural Sparsity
- **163 calendar dates are missing** across the 14-year timeline, concentrated in 2013 (101 days) and 2014 (18 days).
- Shifting must always be date-anchored (using complete calendar date index reindexing) rather than positional shifting.
- Missing values in `holiday_name` (93.5%) represent structural absence of holidays and should be treated as categorical 'None'.

### 8. Critical Considerations for Next-Day Modelling Contract
1. **Operational Horizon ($H=2$):** At the 20:00 IST evening cutoff on Day $D$, Day $D$ demand ($T-1$) is unobserved. The latest available demand observation is Day $D-1$ ($T-2$). Therefore, all demand autoregressive lags and rolling statistics must start at $T-2$ or earlier (`peak_lag_2`).
2. **Weather Forecast vs. Reanalysis:** Target-day weather ($T$) must use day-ahead numerical weather prediction (NWP) forecasts (`weather_forecast_log.csv`), NOT retrospective ERA5 actuals (`weather_daily.csv`), to eliminate data leakage.